# 13 — Optimise — maximise & minimise

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/13_optimise.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/13_optimise.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/13_optimise.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Find the input vector that maximises or minimises any output, with custom bounds and standardised sensitivity at the optimum.

**Mirrors.** **Optimise** tab → *Target*, *Direction*, *k·σ*, *n_starts*, *Bounds* table, *Sensitivity* panel.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Constrained optimisation

Find the input vector `x ∈ [bounds]` that maximises (or minimises) a chosen output. The dashboard uses 8 random starts of an L-BFGS-B search with finite-difference gradients. We mirror that in Python via `cubespec.optimise.optimise()`.


In [ ]:
from cubespec import DEFAULT_CSP, optimise, default_bounds
import pandas as pd

bounds = default_bounds(DEFAULT_CSP, k_sigma=3)
rmax = optimise(DEFAULT_CSP, output='P9_compressive_strength', direction='maximise',
                bounds=bounds, seed=1337, n_starts=8)
rmin = optimise(DEFAULT_CSP, output='P9_compressive_strength', direction='minimise',
                bounds=bounds, seed=1337, n_starts=8)
print(f'P9 max = {rmax.value:.3f} MPa  (converged={rmax.converged}, iters={rmax.iterations})')
print(f'P9 min = {rmin.value:.3f} MPa')


## 2. Where is the optimum, in σ-units of each factor?


In [ ]:
rows = []
for k, (lo, hi) in bounds.items():
    p = DEFAULT_CSP.params[k]
    rows.append({
        'factor': k, 'lower': lo, 'upper': hi,
        'mu':     p.mean,
        'x_max':  rmax.x[k], 'd_max_sigma': (rmax.x[k] - p.mean) / p.sd,
        'x_min':  rmin.x[k], 'd_min_sigma': (rmin.x[k] - p.mean) / p.sd,
    })
df = pd.DataFrame(rows)
df


## 3. Standardised sensitivity at the maximum


In [ ]:
import matplotlib.pyplot as plt
sens = pd.Series({k: rmax.sensitivity[k] * DEFAULT_CSP.params[k].sd for k in bounds})
sens = sens.reindex(sens.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(sens.index, sens.values, color=['#5b8def' if v >= 0 else '#e94f64' for v in sens.values])
ax.axvline(0, color='black', lw=0.6)
ax.set_xlabel('∂P9 / ∂x · σ_x  (MPa per 1-σ change)')
ax.set_title('Standardised sensitivity at the maximiser')
fig.tight_layout(); plt.show()


## 4. Custom bounds — mirrors the editable bounds table

Tighten P4_Fx to ±1σ around its mean and re-run.


In [ ]:
tight = dict(bounds)
p4 = DEFAULT_CSP.params['P4_Fx']
tight['P4_Fx'] = (p4.mean - p4.sd, p4.mean + p4.sd)
rt = optimise(DEFAULT_CSP, output='P9_compressive_strength', direction='maximise',
              bounds=tight, seed=1337, n_starts=8)
print(f'with tighter P4_Fx bounds  P9 max = {rt.value:.3f} MPa  '
      f'(was {rmax.value:.3f}, lost {rmax.value - rt.value:.3f} MPa)')


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
